In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import MobileNet_V2_Weights, VGG16_Weights, VGG19_Weights, Inception_V3_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

ImageFile.LOAD_TRUNCATED_IMAGES = True

COMPUTE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active hardware accelerator: {COMPUTE_DEVICE}")

Active hardware accelerator: cuda


In [4]:
BASE_DATA_DIR = Path("/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/")
ANNOTATIONS_DIR = Path("/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations")

EMBRYO_STAGES = [
    'tPB2', 'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9+',
    'tM', 'tSB', 'tB', 'tEB', 'tHB'
]

stage_to_idx = {stage: index for index, stage in enumerate(EMBRYO_STAGES)}

In [5]:
def generate_dataset_mapping(sampling_interval=6):
    dataset_records = []

    for ann_file in tqdm(list(ANNOTATIONS_DIR.glob("*.csv"))):
        subject_id = ann_file.stem.replace("_phases", "")
        subject_img_dir = BASE_DATA_DIR / subject_id

        if not subject_img_dir.is_dir():
            continue

        available_images = sorted([img.name for img in subject_img_dir.iterdir() if img.is_file()])
        if not available_images:
            continue

        annotation_df = pd.read_csv(ann_file, header=None, names=["phase", "start", "end"])

        for _, record in annotation_df.iterrows():
            target_label = stage_to_idx[record["phase"]]

            for frame_idx in range(record["start"], record["end"], sampling_interval):
                if frame_idx < len(available_images):
                    full_image_path = subject_img_dir / available_images[frame_idx]
                    dataset_records.append({
                        "file_path": str(full_image_path), 
                        "target": target_label, 
                        "subject": subject_id
                    })

    return pd.DataFrame(dataset_records)

master_df = generate_dataset_mapping()
print(f"Extracted {len(master_df)} valid samples.")

  0%|          | 0/704 [00:00<?, ?it/s]

Extracted 52160 valid samples.


In [6]:
unique_subjects = master_df["subject"].unique()

# Split subjects instead of images to prevent data leakage
train_subj, temp_subj = train_test_split(unique_subjects, test_size=0.3, random_state=42)
val_subj, test_subj = train_test_split(temp_subj, test_size=0.5, random_state=42)

df_train = master_df[master_df["subject"].isin(train_subj)].copy()
df_val = master_df[master_df["subject"].isin(val_subj)].copy()
df_test = master_df[master_df["subject"].isin(test_subj)].copy()

# Fix class out of bounds issue by merging the last anomalous class 15 into 14
for split_df in [df_train, df_val, df_test]:
    split_df.loc[split_df["target"] == 15, "target"] = 14

TOTAL_TARGET_CLASSES = 15

print(f"Training split size: {len(df_train)}")
print(f"Validation split size: {len(df_val)}")
print(f"Testing split size: {len(df_test)}")

Training split size: 36402
Validation split size: 7862
Testing split size: 7896


In [7]:
calculated_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(TOTAL_TARGET_CLASSES),
    y=df_train["target"].values
)

balanced_loss_weights = torch.tensor(calculated_weights, dtype=torch.float32).to(COMPUTE_DEVICE)
print(f"Computed Target Weights:\n{balanced_loss_weights}")

Computed Target Weights:
tensor([2.0619, 0.4697, 2.5385, 0.6920, 3.4669, 0.6800, 2.2022, 2.3607, 2.0190,
        0.6158, 0.3862, 1.2244, 1.1426, 1.9539, 1.0020], device='cuda:0')


In [8]:
class EmbryoVisionDataset(Dataset):
    def __init__(self, data_frame, augmentations=None):
        self.data_frame = data_frame.reset_index(drop=True)
        self.augmentations = augmentations

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, index):
        attempts = 0
        while attempts < 5:
            try:
                record = self.data_frame.iloc[index]
                image_data = Image.open(record["file_path"]).convert("RGB")
                image_data = image_data.resize((256, 256))

                if self.augmentations:
                    image_data = self.augmentations(image_data)

                target_tensor = torch.tensor(record["target"], dtype=torch.long)
                return image_data, target_tensor
            except Exception:
                # If image is corrupted, randomly sample another one
                index = np.random.randint(0, len(self.data_frame))
                attempts += 1
                
        # Fallback tensor if all 5 attempts fail
        return torch.zeros(3, 224, 224), torch.tensor(0)

aug_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.CenterCrop(224),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

aug_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.CenterCrop(224)
])

# Build the dataloaders
loader_train = DataLoader(EmbryoVisionDataset(df_train, aug_train), batch_size=32, shuffle=True, num_workers=4)
loader_val   = DataLoader(EmbryoVisionDataset(df_val, aug_eval), batch_size=32, num_workers=4)
loader_test  = DataLoader(EmbryoVisionDataset(df_test, aug_eval), batch_size=32, num_workers=4)

In [9]:
def initialize_network(architecture_name):
    if architecture_name == "mobilenet":
        net = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(net.last_channel, TOTAL_TARGET_CLASSES))
    elif architecture_name == "vgg16":
        net = models.vgg16(weights=VGG16_Weights.DEFAULT)
        net.classifier[6] = nn.Linear(4096, TOTAL_TARGET_CLASSES)
    elif architecture_name == "vgg19":
        net = models.vgg19(weights=VGG19_Weights.DEFAULT)
        net.classifier[6] = nn.Linear(4096, TOTAL_TARGET_CLASSES)
    
    return net.to(COMPUTE_DEVICE)

class FocalOrdinalCrossEntropy(nn.Module):
    def __init__(self, class_weights, num_classes, ordinal_factor=0.5, focal_gamma=2.0):
        super().__init__()
        self.standard_ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        self.ordinal_factor = ordinal_factor
        self.focal_gamma = focal_gamma
        
        # Buffer to keep class indices on the correct GPU/device automatically
        self.register_buffer("class_idx_array", torch.arange(num_classes).view(1, -1))

    def forward(self, predictions, ground_truth):
        probabilities = F.softmax(predictions, dim=1)

        # 1. Base Cross Entropy
        base_loss = self.standard_ce(predictions, ground_truth)

        # 2. Extract probability of the true class and calculate Focal Gate
        batch_indices = torch.arange(predictions.shape[0], device=predictions.device)
        true_class_probs = probabilities[batch_indices, ground_truth]
        focal_multiplier = torch.pow(1.0 - true_class_probs, self.focal_gamma)

        # 3. Cumulative Distribution Function (CDF) for Ordinal ranking penalty
        predicted_cdf = torch.cumsum(probabilities, dim=1)
        actual_cdf = (self.class_idx_array >= ground_truth.unsqueeze(1)).float()

        ordinal_deviation = torch.mean(torch.square(predicted_cdf - actual_cdf), dim=1)

        # Combine
        return base_loss + self.ordinal_factor * torch.mean(focal_multiplier * ordinal_deviation)

In [10]:
def execute_training_epoch(net, dataloader, opt, criteria):
    net.train()
    running_loss = 0.0
    tracked_preds, tracked_actuals = [], []

    for batch_idx, (images, targets) in enumerate(dataloader):
        images, targets = images.to(COMPUTE_DEVICE), targets.to(COMPUTE_DEVICE)

        opt.zero_grad()
        outputs = net(images)
        loss_val = criteria(outputs, targets)
        
        loss_val.backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        opt.step()

        running_loss += loss_val.item()

        inferred_classes = torch.argmax(outputs, dim=1)
        tracked_preds.extend(inferred_classes.detach().cpu().numpy())
        tracked_actuals.extend(targets.cpu().numpy())

        if batch_idx % 100 == 0:
            current_acc = (inferred_classes == targets).float().mean().item()
            print(f"Iteration {batch_idx}/{len(dataloader)} | Step Loss: {loss_val.item():.4f} | Step Acc: {current_acc:.4f}")

    overall_accuracy = accuracy_score(tracked_actuals, tracked_preds)
    return running_loss / len(dataloader), overall_accuracy

def validate_model(net, dataloader):
    net.eval()
    inferences, actuals = [], []

    with torch.no_grad():
        for images, targets in dataloader:
            outputs = net(images.to(COMPUTE_DEVICE))
            chosen_classes = torch.argmax(outputs, dim=1).cpu().numpy()
            
            inferences.extend(chosen_classes)
            actuals.extend(targets.numpy())

    return accuracy_score(actuals, inferences), f1_score(actuals, inferences, average="weighted")


# ---------------- MAIN RUNNER ---------------- 
model_benchmarks = {}
networks_to_train = ["mobilenet", "vgg16", "vgg19"]

for arch in networks_to_train:
    print(f"\n==== INITIALIZING {arch.upper()} ====")
    current_model = initialize_network(arch)
    optimizer_func = torch.optim.Adam(current_model.parameters(), lr=1e-4, weight_decay=1e-4)
    loss_calculator = FocalOrdinalCrossEntropy(balanced_loss_weights, TOTAL_TARGET_CLASSES).to(COMPUTE_DEVICE)

    for epoch_num in range(5):
        t_loss, t_acc = execute_training_epoch(current_model, loader_train, optimizer_func, loss_calculator)
        v_acc, v_f1 = validate_model(current_model, loader_val)

        print(f"--> Epoch {epoch_num} | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f} | Val F1: {v_f1:.4f}")

    final_test_acc, final_test_f1 = validate_model(current_model, loader_test)
    model_benchmarks[arch] = (final_test_acc, final_test_f1)

    print(f"\n[FINAL SCORES] {arch.upper()} -> Test Accuracy: {final_test_acc:.4f}, Test F1: {final_test_f1:.4f}")


==== INITIALIZING MOBILENET ====
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 116MB/s] 


Iteration 0/1138 | Step Loss: 3.0021 | Step Acc: 0.1250
Iteration 100/1138 | Step Loss: 2.7765 | Step Acc: 0.0938
Iteration 200/1138 | Step Loss: 2.5438 | Step Acc: 0.2188
Iteration 300/1138 | Step Loss: 2.4574 | Step Acc: 0.1875
Iteration 400/1138 | Step Loss: 2.3668 | Step Acc: 0.3750
Iteration 500/1138 | Step Loss: 2.5135 | Step Acc: 0.0938
Iteration 600/1138 | Step Loss: 2.4327 | Step Acc: 0.2188
Iteration 700/1138 | Step Loss: 2.3476 | Step Acc: 0.3125
Iteration 800/1138 | Step Loss: 2.6307 | Step Acc: 0.1875
Iteration 900/1138 | Step Loss: 2.3393 | Step Acc: 0.3125
Iteration 1000/1138 | Step Loss: 2.5166 | Step Acc: 0.0938
Iteration 1100/1138 | Step Loss: 1.9580 | Step Acc: 0.4375
--> Epoch 0 | Train Loss: 2.4084 | Train Acc: 0.2653 | Val Acc: 0.2840 | Val F1: 0.2939
Iteration 0/1138 | Step Loss: 2.1471 | Step Acc: 0.3125
Iteration 100/1138 | Step Loss: 2.1731 | Step Acc: 0.4688
Iteration 200/1138 | Step Loss: 2.1547 | Step Acc: 0.2500
Iteration 300/1138 | Step Loss: 2.3652 | Ste

100%|██████████| 528M/528M [00:02<00:00, 199MB/s]  


Iteration 0/1138 | Step Loss: 2.9818 | Step Acc: 0.0938
Iteration 100/1138 | Step Loss: 2.5417 | Step Acc: 0.2188
Iteration 200/1138 | Step Loss: 2.5440 | Step Acc: 0.1250
Iteration 300/1138 | Step Loss: 3.0593 | Step Acc: 0.2812
Iteration 400/1138 | Step Loss: 2.8668 | Step Acc: 0.0938
Iteration 500/1138 | Step Loss: 2.3224 | Step Acc: 0.3125
Iteration 600/1138 | Step Loss: 2.5377 | Step Acc: 0.2188
Iteration 700/1138 | Step Loss: 2.4177 | Step Acc: 0.3750
Iteration 800/1138 | Step Loss: 2.1999 | Step Acc: 0.3125
Iteration 900/1138 | Step Loss: 2.5521 | Step Acc: 0.2500
Iteration 1000/1138 | Step Loss: 2.1421 | Step Acc: 0.4375
Iteration 1100/1138 | Step Loss: 2.3038 | Step Acc: 0.2188
--> Epoch 0 | Train Loss: 2.4379 | Train Acc: 0.2403 | Val Acc: 0.2689 | Val F1: 0.2741
Iteration 0/1138 | Step Loss: 2.1831 | Step Acc: 0.3125
Iteration 100/1138 | Step Loss: 2.1655 | Step Acc: 0.2500
Iteration 200/1138 | Step Loss: 2.1460 | Step Acc: 0.2500
Iteration 300/1138 | Step Loss: 2.1312 | Ste

100%|██████████| 548M/548M [00:03<00:00, 179MB/s]  


Iteration 0/1138 | Step Loss: 2.9448 | Step Acc: 0.0938
Iteration 100/1138 | Step Loss: 2.5918 | Step Acc: 0.2188
Iteration 200/1138 | Step Loss: 2.4559 | Step Acc: 0.2812
Iteration 300/1138 | Step Loss: 2.3141 | Step Acc: 0.2500
Iteration 400/1138 | Step Loss: 2.6161 | Step Acc: 0.2500
Iteration 500/1138 | Step Loss: 2.3336 | Step Acc: 0.2188
Iteration 600/1138 | Step Loss: 2.4099 | Step Acc: 0.1875
Iteration 700/1138 | Step Loss: 2.4379 | Step Acc: 0.1875
Iteration 800/1138 | Step Loss: 2.1261 | Step Acc: 0.4375
Iteration 900/1138 | Step Loss: 2.2354 | Step Acc: 0.3438
Iteration 1000/1138 | Step Loss: 2.1997 | Step Acc: 0.2188
Iteration 1100/1138 | Step Loss: 2.4177 | Step Acc: 0.2500
--> Epoch 0 | Train Loss: 2.4377 | Train Acc: 0.2350 | Val Acc: 0.2892 | Val F1: 0.2950
Iteration 0/1138 | Step Loss: 2.3045 | Step Acc: 0.2812
Iteration 100/1138 | Step Loss: 2.1906 | Step Acc: 0.2500
Iteration 200/1138 | Step Loss: 2.1658 | Step Acc: 0.3750
Iteration 300/1138 | Step Loss: 2.1084 | Ste

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import Inception_V3_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# Allow truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

COMPUTE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active hardware accelerator: {COMPUTE_DEVICE}")

Active hardware accelerator: cuda


In [12]:
BASE_DATA_DIR = Path("/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/")
ANNOTATIONS_DIR = Path("/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations")

EMBRYO_STAGES = [
    'tPB2', 'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9+',
    'tM', 'tSB', 'tB', 'tEB', 'tHB'
]
stage_to_idx = {stage: index for index, stage in enumerate(EMBRYO_STAGES)}

def generate_dataset_mapping(sampling_interval=6):
    dataset_records = []
    for ann_file in tqdm(list(ANNOTATIONS_DIR.glob("*.csv"))):
        subject_id = ann_file.stem.replace("_phases", "")
        subject_img_dir = BASE_DATA_DIR / subject_id

        if not subject_img_dir.is_dir():
            continue

        available_images = sorted([img.name for img in subject_img_dir.iterdir() if img.is_file()])
        if not available_images:
            continue

        annotation_df = pd.read_csv(ann_file, header=None, names=["phase", "start", "end"])
        for _, record in annotation_df.iterrows():
            target_label = stage_to_idx[record["phase"]]
            for frame_idx in range(record["start"], record["end"], sampling_interval):
                if frame_idx < len(available_images):
                    full_image_path = subject_img_dir / available_images[frame_idx]
                    dataset_records.append({"file_path": str(full_image_path), "target": target_label, "subject": subject_id})
    return pd.DataFrame(dataset_records)

master_df = generate_dataset_mapping()

# Subject-wise splitting
unique_subjects = master_df["subject"].unique()
train_subj, temp_subj = train_test_split(unique_subjects, test_size=0.3, random_state=42)
val_subj, test_subj = train_test_split(temp_subj, test_size=0.5, random_state=42)

df_train = master_df[master_df["subject"].isin(train_subj)].copy()
df_val = master_df[master_df["subject"].isin(val_subj)].copy()
df_test = master_df[master_df["subject"].isin(test_subj)].copy()

# Fix bounds logic
for split_df in [df_train, df_val, df_test]:
    split_df.loc[split_df["target"] == 15, "target"] = 14

TOTAL_TARGET_CLASSES = 15

# Class weights
calculated_weights = compute_class_weight(class_weight='balanced', classes=np.arange(TOTAL_TARGET_CLASSES), y=df_train["target"].values)
balanced_loss_weights = torch.tensor(calculated_weights, dtype=torch.float32).to(COMPUTE_DEVICE)

print(f"Train size: {len(df_train)} | Val size: {len(df_val)} | Test size: {len(df_test)}")

  0%|          | 0/704 [00:00<?, ?it/s]

Train size: 36402 | Val size: 7862 | Test size: 7896


In [19]:
class EmbryoVisionDataset(Dataset):
    def __init__(self, data_frame, augmentations=None):
        self.data_frame = data_frame.reset_index(drop=True)
        self.augmentations = augmentations

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, index):
        attempts = 0
        while attempts < 5:
            try:
                record = self.data_frame.iloc[index]
                image_data = Image.open(record["file_path"]).convert("RGB")
                
                if self.augmentations:
                    image_data = self.augmentations(image_data)

                target_tensor = torch.tensor(record["target"], dtype=torch.long)
                return image_data, target_tensor
            except Exception:
                index = np.random.randint(0, len(self.data_frame))
                attempts += 1
                
        # Fallback tensor sized for Inception (299x299) if all attempts fail
        return torch.zeros(3, 299, 299), torch.tensor(0)

# 299x299 is explicitly required for Inception
aug_train_inception = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize(342), 
    transforms.CenterCrop(299), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

aug_eval_inception = transforms.Compose([
    transforms.Resize(342),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

loader_train_inc = DataLoader(EmbryoVisionDataset(df_train, aug_train_inception), batch_size=32, shuffle=True, num_workers=4)
loader_val_inc   = DataLoader(EmbryoVisionDataset(df_val, aug_eval_inception), batch_size=32, num_workers=4)
loader_test_inc  = DataLoader(EmbryoVisionDataset(df_test, aug_eval_inception), batch_size=32, num_workers=4)

In [21]:
class FocalOrdinalCrossEntropy(nn.Module):
    def __init__(self, class_weights, num_classes, ordinal_factor=0.5, focal_gamma=2.0):
        super().__init__()
        self.standard_ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        self.ordinal_factor = ordinal_factor
        self.focal_gamma = focal_gamma
        self.register_buffer("class_idx_array", torch.arange(num_classes).view(1, -1))

    def forward(self, predictions, ground_truth):
        probabilities = F.softmax(predictions, dim=1)
        base_loss = self.standard_ce(predictions, ground_truth)

        batch_indices = torch.arange(predictions.shape[0], device=predictions.device)
        true_class_probs = probabilities[batch_indices, ground_truth]
        focal_multiplier = torch.pow(1.0 - true_class_probs, self.focal_gamma)

        predicted_cdf = torch.cumsum(probabilities, dim=1)
        actual_cdf = (self.class_idx_array >= ground_truth.unsqueeze(1)).float()
        ordinal_deviation = torch.mean(torch.square(predicted_cdf - actual_cdf), dim=1)

        return base_loss + self.ordinal_factor * torch.mean(focal_multiplier * ordinal_deviation)

def get_inception_network():
    # Load the model with default weights (it will load aux weights automatically)
    net = models.inception_v3(weights=Inception_V3_Weights.DEFAULT)
    
    # Disable aux_logits AFTER initialization to prevent output tuple errors
    net.aux_logits = False 
    
    # Replace the fully connected layer
    net.fc = nn.Linear(net.fc.in_features, TOTAL_TARGET_CLASSES)
    return net.to(COMPUTE_DEVICE)

In [22]:
def execute_training_epoch(net, dataloader, opt, criteria):
    net.train()
    running_loss = 0.0
    tracked_preds, tracked_actuals = [], []

    for batch_idx, (images, targets) in enumerate(dataloader):
        images, targets = images.to(COMPUTE_DEVICE), targets.to(COMPUTE_DEVICE)

        opt.zero_grad()
        outputs = net(images)
        loss_val = criteria(outputs, targets)
        
        loss_val.backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        opt.step()

        running_loss += loss_val.item()

        inferred_classes = torch.argmax(outputs, dim=1)
        tracked_preds.extend(inferred_classes.detach().cpu().numpy())
        tracked_actuals.extend(targets.cpu().numpy())

        if batch_idx % 100 == 0:
            current_acc = (inferred_classes == targets).float().mean().item()
            print(f"Iteration {batch_idx}/{len(dataloader)} | Step Loss: {loss_val.item():.4f} | Step Acc: {current_acc:.4f}")

    overall_accuracy = accuracy_score(tracked_actuals, tracked_preds)
    return running_loss / len(dataloader), overall_accuracy

def validate_model(net, dataloader):
    net.eval()
    inferences, actuals = [], []

    with torch.no_grad():
        for images, targets in dataloader:
            outputs = net(images.to(COMPUTE_DEVICE))
            chosen_classes = torch.argmax(outputs, dim=1).cpu().numpy()
            
            inferences.extend(chosen_classes)
            actuals.extend(targets.numpy())

    return accuracy_score(actuals, inferences), f1_score(actuals, inferences, average="weighted")


# ---------------- RUN ISOLATED INCEPTION ---------------- 
print("\n==== INITIALIZING INCEPTION_V3 ====")
inception_model = get_inception_network()
optimizer_func = torch.optim.Adam(inception_model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_calculator = FocalOrdinalCrossEntropy(balanced_loss_weights, TOTAL_TARGET_CLASSES).to(COMPUTE_DEVICE)

for epoch_num in range(5):
    t_loss, t_acc = execute_training_epoch(inception_model, loader_train_inc, optimizer_func, loss_calculator)
    v_acc, v_f1 = validate_model(inception_model, loader_val_inc)

    print(f"--> Epoch {epoch_num} | Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f} | Val F1: {v_f1:.4f}")

final_test_acc, final_test_f1 = validate_model(inception_model, loader_test_inc)
print(f"\n[FINAL SCORES] INCEPTION_V3 -> Test Accuracy: {final_test_acc:.4f}, Test F1: {final_test_f1:.4f}")


==== INITIALIZING INCEPTION_V3 ====
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 182MB/s]  


Iteration 0/1138 | Step Loss: 2.8812 | Step Acc: 0.1250
Iteration 100/1138 | Step Loss: 2.2591 | Step Acc: 0.3125
Iteration 200/1138 | Step Loss: 2.3983 | Step Acc: 0.4062
Iteration 300/1138 | Step Loss: 2.1987 | Step Acc: 0.1875
Iteration 400/1138 | Step Loss: 2.3033 | Step Acc: 0.3438
Iteration 500/1138 | Step Loss: 2.1241 | Step Acc: 0.5000
Iteration 600/1138 | Step Loss: 2.3005 | Step Acc: 0.3125
Iteration 700/1138 | Step Loss: 1.9674 | Step Acc: 0.3750
Iteration 800/1138 | Step Loss: 2.2093 | Step Acc: 0.2500
Iteration 900/1138 | Step Loss: 2.1950 | Step Acc: 0.3125
Iteration 1000/1138 | Step Loss: 2.1103 | Step Acc: 0.3125
Iteration 1100/1138 | Step Loss: 1.9665 | Step Acc: 0.4688
--> Epoch 0 | Train Loss: 2.3125 | Train Acc: 0.2826 | Val Acc: 0.3111 | Val F1: 0.3179
Iteration 0/1138 | Step Loss: 2.0421 | Step Acc: 0.4062
Iteration 100/1138 | Step Loss: 2.0468 | Step Acc: 0.4062
Iteration 200/1138 | Step Loss: 2.1505 | Step Acc: 0.3125
Iteration 300/1138 | Step Loss: 2.2787 | Ste

## 1. Proof of Piecewise Differentiability

**Objective:** Prove the derivative exists everywhere in the operational domain.

Let the loss be $L(p) = -y \log(p) - (1-y)\log(1-p)$, where $y \in \{0, 1\}$ is the true label and $p \in (0, 1)$ is the predicted probability.

We compute the first derivative with respect to $p$:

$$\frac{dL}{dp} = -\frac{y}{p} + \frac{1-y}{1-p}$$

**Proof:** Because neural networks use activation functions (like Sigmoid or Softmax) that bound outputs to the open interval $p \in (0, 1)$, $p$ is never exactly $0$ and $1-p$ is never exactly $0$. Therefore, the denominators are never zero. The derivative is finite and continuous everywhere in $(0, 1)$, satisfying the requirement for gradient descent. $\blacksquare$

---

## 2. Proof of Monotonicity

**Objective:** Prove the loss strictly decreases as $p$ approaches $y$.

Assume the true class is positive ($y = 1$). The loss simplifies to:

$$L(p) = -\log(p)$$

**Proof:** We evaluate the sign of the first derivative:

$$\frac{dL}{dp} = -\frac{1}{p}$$

Since the predicted probability $p$ is strictly positive ($p > 0$), the fraction $\frac{1}{p}$ is positive, making $-\frac{1}{p}$ strictly negative.

Because $\frac{dL}{dp} < 0$ for all $p \in (0, 1)$, the function is strictly monotonically decreasing. It never increases as $p \to 1$. $\blacksquare$

---

## 3. Proof of Targeted Convergence (Focal vs. Standard)

**Objective:** Prove that advanced loss functions decay gradients for "easy" examples to target "hard" examples.

Let's compute the derivative of the loss with respect to the raw network logit $z$, where $p = \sigma(z)$ is the sigmoid activation, assuming $y = 1$.

**For Standard Cross-Entropy:**

$$\frac{dL_{\text{CE}}}{dz} = p - 1$$

As $p \to 1$ (an easy example), the gradient approaches $0$, but slowly — linearly. Millions of small gradients from easy examples can still overwhelm the network.

**For Focal Loss** $L_{\text{FL}} = -(1-p)^{\gamma} \log(p)$, where $\gamma > 0$ is the focusing parameter:

$$\frac{dL_{\text{FL}}}{dz} = (1-p)^{\gamma} \Big( \gamma p \log(p) + p - 1 \Big)$$

**Proof:** Because of the $(1-p)^{\gamma}$ modulating factor, as $p \to 1$, the gradient decays *exponentially* faster to $0$ (for $\gamma > 1$). This mathematically guarantees that easy examples contribute negligibly to the optimizer's momentum, forcing the gradient signal to be exclusively populated by misclassified (hard) examples, resulting in faster and more targeted convergence. $\blacksquare$

---

## 4. Proof of Numerical Stability (LogSumExp Trick)

**Objective:** Prove the loss calculation is bounded and cannot produce `NaN` or `Inf` from floating-point overflow.

If a network is severely wrong, it might output a highly negative logit, e.g., $z = -1000$.

The standard calculation gives:

$$L(z) = -\log(\sigma(z)) = -\log\!\left(\frac{1}{1 + e^{-z}}\right) = \log(1 + e^{-z})$$

If evaluated directly, $e^{-(-1000)} = e^{1000}$, which exceeds 32-bit float limits and becomes `Infinity`. Then $\log(\infty)$ yields `NaN`, crashing the model.

**Proof:** Stable loss functions rewrite this using the identity:

$$\log(1 + e^{-z}) \;\equiv\; \max(0,\,-z) + \log\!\left(1 + e^{-|z|}\right)$$

Evaluating at $z = -1000$:

| Term | Computation | Result |
|---|---|---|
| $\max(0,\,-z)$ | $\max(0, 1000)$ | $1000$ — safe linear float |
| $e^{-\vert z \vert}$ | $e^{-1000}$ | $\approx 0$ |
| $\log(1 + e^{-\vert z \vert})$ | $\log(1 + 0) = \log(1)$ | $0$ |
| **Total Loss** | $1000 + 0$ | $\mathbf{1000}$ |

Since the exponent argument is always $-|z| \leq 0$, we have $e^{-|z|} \in [0, 1]$ for all $z \in \mathbb{R}$. This strictly bounds the inner term, proving the function can **never** overflow. Numerical stability is guaranteed. $\blacksquare$